# 10. 밝기와 명암비

화소 처리, 포화 연산, 영상 산술, 히스토그램과 HSV 색상 추출을 실습합니다.

> 이미지 예제는 노트북과 같은 위치에 `data` 폴더를 만들고 강의에서 사용하는 파일을 넣어 실행하세요.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_image(name, flags=cv2.IMREAD_COLOR):
    path = Path('data') / name
    image = cv2.imread(str(path), flags)
    if image is None:
        raise FileNotFoundError(path)
    return image

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        if image.ndim == 2:
            ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()

## 밝기 조절과 uint8 오버플로

In [ ]:
gray = read_image('lenna.bmp', cv2.IMREAD_GRAYSCALE)
bright_cv = cv2.add(gray, 100)
bright_clip = np.clip(gray.astype(np.float32) + 100, 0, 255).astype(np.uint8)
overflow = gray + np.uint8(100)  # 비교용: 255를 넘으면 wrap-around
show([gray, bright_cv, bright_clip, overflow], ['source', 'cv2.add', 'np.clip', 'uint8 overflow'])

In [ ]:
color = read_image('lenna.bmp')
color_bright = cv2.add(color, (100, 100, 100, 0))
show([color, color_bright], ['source', 'BGR channels +100'])

## 영상의 덧셈·뺄셈·차이

In [ ]:
src1 = read_image('lenna.bmp')
src2 = read_image('sky.bmp')
src2 = cv2.resize(src2, (src1.shape[1], src1.shape[0]))
blend = cv2.addWeighted(src1, 0.5, src2, 0.5, 0)
show([src1, src2, blend], ['source 1', 'source 2', 'weighted sum'])

In [ ]:
road1 = read_image('road1.jpg', cv2.IMREAD_GRAYSCALE)
road2 = read_image('road2.jpg', cv2.IMREAD_GRAYSCALE)
difference = cv2.absdiff(road1, road2)
_, motion_mask = cv2.threshold(difference, 30, 255, cv2.THRESH_BINARY)
show([road1, road2, difference, motion_mask], ['frame 1', 'frame 2', 'absdiff', 'threshold mask'])

## 명암비와 히스토그램 스트레칭

In [ ]:
alpha = 1.0
contrast = np.clip((1 + alpha) * gray.astype(np.float32) - 128 * alpha, 0, 255).astype(np.uint8)
show([gray, contrast], ['source', 'contrast around 128'])

In [ ]:
hawkes = read_image('hawkes.png', cv2.IMREAD_GRAYSCALE)
stretched = cv2.normalize(hawkes, None, 0, 255, cv2.NORM_MINMAX)
equalized = cv2.equalizeHist(hawkes)
show([hawkes, stretched, equalized], ['source', 'stretching', 'equalization'])

## 컬러 영상 평활화: Y 채널만 처리

In [ ]:
field = read_image('field.bmp')
field_ycrcb = cv2.cvtColor(field, cv2.COLOR_BGR2YCrCb)
y, cr, cb = cv2.split(field_ycrcb)
y_equalized = cv2.equalizeHist(y)
field_result = cv2.cvtColor(cv2.merge([y_equalized, cr, cb]), cv2.COLOR_YCrCb2BGR)
show([field, field_result], ['source', 'Y-channel equalization'])

## HSV 범위로 특정 색상 추출

In [ ]:
candies = read_image('candies.png')
candies_hsv = cv2.cvtColor(candies, cv2.COLOR_BGR2HSV)
green_mask = cv2.inRange(candies_hsv, (50, 150, 0), (80, 255, 255))
green_only = cv2.bitwise_and(candies, candies, mask=green_mask)
show([candies, green_mask, green_only], ['source', 'HSV mask', 'green objects'])